In [41]:
"""yashaswini MG
24UG00342"""

In [45]:
import pandas as pd
import json

def Beta_load_and_integrate():
    folder_path = "/content/"

    "Load all datasets"
    df1 = pd.read_json(folder_path + "auxiliary_metadata.json")
    df2 = pd.read_csv(folder_path + "class.csv")
    df3 = pd.read_csv(folder_path + "zoo.csv")

    print("All csv's loaded")

    "Convert animal names to lowercase across all datasets"
    if "animal_name" in df1.columns:
        df1["animal_name"] = df1["animal_name"].astype(str).str.lower()

    if "animal_name" in df2.columns:
        df2["animal_name"] = df2["animal_name"].astype(str).str.lower()

    if "animal_name" in df3.columns:
        df3["animal_name"] = df3["animal_name"].astype(str).str.lower()

    "Standardize column names in JSON data"
    df1 = df1.rename(columns={
        "conservation": "conservation_status",
        "status": "conservation_status",
        "habitat": "habitat_type",
        "habitats": "habitat_type",
        "diet_type": "diet"
    })


    def safe_stringify_column(df, column_name):
        if column_name in df.columns:
            "Convert entire column to string safely"
            df[column_name] = df[column_name].apply(
                lambda x: json.dumps(x) if isinstance(x, (dict, list)) else str(x)
            )
        return df

    "Process and clean auxiliary data columns using the safe function"
    for column in ["animal_name", "habitat_type", "diet", "conservation_status"]:
        if column in df1.columns:
            df1 = safe_stringify_column(df1, column)

            df1[column] = df1[column].str.lower()

    "Fix typos in diet column"
    if "diet" in df1.columns:
        df1["diet"] = df1["diet"].replace({
            "omnivor": "omnivore",
            "carnivor": "carnivore",
            "herbivor": "herbivore"
        })

    "Normalize habitat values"
    if "habitat_type" in df1.columns:
        df1["habitat_type"] = df1["habitat_type"].replace({
            "fresh water": "freshwater",
            "fresh-water": "freshwater",
            "freshwater": "freshwater",
            "forest": "terrestrial",
            "grasslands": "terrestrial",
            "savanna": "terrestrial",
            "domestic": "terrestrial",
            "marine/coastal": "marine",
            "marine": "marine",
            "terrestrial": "terrestrial"
        })

    " Merge all datasets"
    merged_df = pd.merge(df1, df2, on="animal_name", how="left")
    merged_df = pd.merge(merged_df, df3, on="animal_name", how="left")

    """FEATURE ENGINEERING - Add numerical scores for habitat and diet
    Habitat scoring: marine=3, freshwater=2, terrestrial=1"""
    if "habitat_type" in merged_df.columns:
        merged_df["habitat_score"] = merged_df["habitat_type"].map({
            "marine": 3,
            "freshwater": 2,
            "terrestrial": 1
        })
        " Fill missing values with 1 (terrestrial as default)"
        merged_df["habitat_score"] = merged_df["habitat_score"].fillna(1)

    "Diet scoring: carnivore=3, omnivore=2, herbivore=1, others=1"
    if "diet" in merged_df.columns:

        def get_diet_score(diet_str):
            diet_str = str(diet_str).lower()
            if "carnivore" in diet_str:
                return 3
            elif "omnivore" in diet_str:
                return 2
            elif "herbivore" in diet_str:
                return 1
            else:
                return 1

        merged_df["diet_score"] = merged_df["diet"].apply(get_diet_score)


    if "habitat_score" in merged_df.columns and "diet_score" in merged_df.columns:
        merged_df["ecological_score"] = merged_df["habitat_score"] + merged_df["diet_score"]
        print("Added ecological_score feature (habitat_score + diet_score)")


    print("Rows before dropping missing auxiliary fields:", len(merged_df))


# New Section